# AI vs Real Detector — Colab Training + Branch Ablation

Trains on the CNNDetection + FFHQ + LSUN split prepared by `dataset_prep_RUN2_combined.ipynb`. The data stays on Google Drive/Colab; this notebook never downloads it locally.

The model scores **general AI-generated images**, not faces only. Physics features are scene-level (illumination, spectrum, compression, color, texture).

This notebook trains **five independent models** on the same data, then reports held-out test metrics:

1. Deep branch only (EfficientNet)
2. Physics branch only
3. PRNU branch only
4. Semantic / ViT branch only
5. All four branches combined (`full_hybrid`)

The table reports the measured result; only claim the combined model is stronger if its held-out-test metrics exceed every single branch.

**GPU:** T4 or better. Prefer copying this repo from Drive so unpublished local changes are used.


In [ ]:
# 1. GPU check
import torch

assert torch.cuda.is_available(), "Enable Runtime → Change runtime type → GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)


In [ ]:
# 2. Mount Drive, then use Drive code if present (else clone GitHub)
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil

%cd /content

DRIVE_ROOT = Path("/content/drive/MyDrive/ai-vs-real-face-detector")
DRIVE_CODE = DRIVE_ROOT / "code"
PROJECT = Path("/content/ai-vs-real-face-detector")

if (DRIVE_CODE / "src" / "train.py").exists():
    shutil.rmtree(PROJECT, ignore_errors=True)
    shutil.copytree(DRIVE_CODE, PROJECT)
    print("Using project copy from Drive:", DRIVE_CODE)
else:
    !rm -rf /content/ai-vs-real-face-detector
    !git clone --branch master --single-branch https://github.com/Algorithm-bot/ai-vs-real-face-detector.git /content/ai-vs-real-face-detector
    nested = PROJECT / "ai-vs-real-face-detector"
    if (nested / "src" / "train.py").exists():
        PROJECT = nested
    print("Cloned GitHub. To use unpublished local edits, copy the repo to", DRIVE_CODE)

print("PROJECT:", PROJECT)
!grep -n "physics_only" "{PROJECT}/src/train.py" | head


In [ ]:
# 3. Data and checkpoint paths on Drive
from pathlib import Path
import json

DATA_DIR = DRIVE_ROOT / "data"
OUTPUT_ROOT = DRIVE_ROOT / "models" / "cnn_detection_ablation"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

REQUIRE_OFFICIAL_PROTOCOL = True
manifest_path = DATA_DIR / "cnndetection_manifest.json"
if manifest_path.exists():
    dataset_manifest = json.loads(manifest_path.read_text())
    print("Dataset protocol:", dataset_manifest.get("protocol", "legacy manifest"))
    if REQUIRE_OFFICIAL_PROTOCOL:
        assert dataset_manifest.get("protocol") == "official train/val/test", (
            "Rebuild the data with DOWNLOAD_FULL_TRAINSET=True, or set REQUIRE_OFFICIAL_PROTOCOL=False for a smoke test."
        )
else:
    print("WARNING: no manifest found; dataset provenance cannot be verified.")

print("DATA:", DATA_DIR)
print("OUTPUT:", OUTPUT_ROOT)


In [ ]:
# 4. Install project dependencies
%cd {PROJECT}
!pip -q install -r requirements.txt


In [ ]:
# 5. Verify CNNDetection-style train/val/test
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def count_images(path):
    return sum(
        1 for p in Path(path).rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

counts = {}
for split in ["train", "val", "test"]:
    counts[split] = {}
    for label in ["real", "fake"]:
        p = DATA_DIR / split / label
        assert p.exists(), f"Missing dataset directory: {p}. Run the dataset-prep notebook first."
        counts[split][label] = count_images(p)
        print(f"{split}/{label}: {counts[split][label]}")
        assert counts[split][label] > 0, f"No images in {p}"

assert counts["train"] != counts["val"], "Train and validation counts are unexpectedly identical; verify the dataset build."
print("Dataset structure OK. The test split is held out from training and validation.")
manifest = DATA_DIR / "cnndetection_manifest.json"
if manifest.exists():
    print(manifest.read_text()[:2000])


## 6. Shared training settings

Train each ablation with the same epochs and seed so the comparison is fair. Optional 1-epoch smoke test is off by default.


In [ ]:
EPOCHS = 12
BATCH_FULL = 16
BATCH_BRANCH = 32
WORKERS = 2
SEED = 42
RUN_SMOKE = False


In [ ]:
# Optional smoke test: full hybrid, 1 epoch
%cd {PROJECT}
if RUN_SMOKE:
    !python src/train.py --mode full_hybrid --data-dir "{DATA_DIR}" --output-dir "{OUTPUT_ROOT}/smoke_test" --epochs 1 --batch-size 2 --num-workers 0 --fusion-mode gated --seed {SEED}
else:
    print("Skipping smoke test. Set RUN_SMOKE = True to enable.")


## 7. Train the five models

Each cell writes a checkpoint under `cnn_detection_ablation/<mode>/`.


In [ ]:
# 7a. Deep branch only
%cd {PROJECT}
!python src/train.py --mode stage1 --data-dir "{DATA_DIR}" --output-dir "{OUTPUT_ROOT}/stage1" --epochs {EPOCHS} --batch-size {BATCH_FULL} --num-workers {WORKERS} --seed {SEED}


In [ ]:
# 7b. Physics branch only
%cd {PROJECT}
!python src/train.py --mode physics_only --data-dir "{DATA_DIR}" --output-dir "{OUTPUT_ROOT}/physics_only" --epochs {EPOCHS} --batch-size {BATCH_BRANCH} --num-workers {WORKERS} --seed {SEED}


In [ ]:
# 7c. PRNU branch only
%cd {PROJECT}
!python src/train.py --mode prnu_only --data-dir "{DATA_DIR}" --output-dir "{OUTPUT_ROOT}/prnu_only" --epochs {EPOCHS} --batch-size {BATCH_BRANCH} --num-workers {WORKERS} --seed {SEED}


In [ ]:
# 7d. Semantic / ViT branch only
%cd {PROJECT}
!python src/train.py --mode semantic_only --data-dir "{DATA_DIR}" --output-dir "{OUTPUT_ROOT}/semantic_only" --epochs {EPOCHS} --batch-size {BATCH_BRANCH} --num-workers {WORKERS} --seed {SEED}


In [ ]:
# 7e. All four branches combined
%cd {PROJECT}
!python src/train.py --mode full_hybrid --data-dir "{DATA_DIR}" --output-dir "{OUTPUT_ROOT}/full_hybrid" --epochs {EPOCHS} --batch-size {BATCH_FULL} --num-workers {WORKERS} --fusion-mode gated --seed {SEED}


## 8. Held-out test metrics (ablation table)

Evaluates each trained checkpoint on the same `data/test` split and writes `ablation_metrics.json`.


In [ ]:
# 8. Compare independently trained branches on the official test split
%cd {PROJECT}
EVAL_DIR = OUTPUT_ROOT / "ablation_eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

!python src/evaluate.py --ablation --checkpoint "{OUTPUT_ROOT}" --data-dir "{DATA_DIR}" --output-dir "{EVAL_DIR}"


In [ ]:
# 9. Pretty-print the comparison table
import pandas as pd

metrics_path = OUTPUT_ROOT / "ablation_eval" / "ablation_metrics.json"
assert metrics_path.exists(), metrics_path
payload = json.loads(metrics_path.read_text())
df = pd.DataFrame(payload["comparison"])
order = ["deep_only", "physics_only", "prnu_only", "semantic_only", "full"]
df["variant"] = pd.Categorical(df["variant"], categories=order, ordered=True)
df = df.sort_values("variant")
display(df)
print("\nInterpretation: compare the measured held-out accuracy/F1/AUC; do not assume the full model wins.")
print(json.dumps(payload["comparison"], indent=2))

# Generator/source-wise accuracy makes the general-image claim auditable.
generator_rows = []
for variant, result in payload["variants"].items():
    groups = result.get("by_generator") or result.get("by_source", {})
    for source, metric in groups.items():
        generator_rows.append({
            "variant": variant, "source": source, "samples": metric.get("samples"),
            "accuracy": metric.get("accuracy"), "f1_score": metric.get("f1_score"),
            "roc_auc": metric.get("roc_auc"),
        })
if generator_rows:
    display(pd.DataFrame(generator_rows).sort_values(["source", "variant"]))


In [ ]:
# 10. List checkpoints
for p in sorted(OUTPUT_ROOT.rglob("*_best.pt")):
    print(p, p.stat().st_size)


# Done

Checkpoints live on Drive under `models/cnn_detection_ablation/`.

Cite held-out-test accuracy / F1 / ROC-AUC from `ablation_metrics.json` when arguing that physics, PRNU, semantics, and the CNN backbone are complementary. Include the source-wise table, not just the aggregate, when making a general-image claim.
